In [1]:
# ============================================================
# RETRIEVAL AUGMENTED GENERATION (RAG) SYSTEM
# ============================================================

# Install Required Libraries
!pip install -q transformers sentence-transformers faiss-cpu torch sentencepiece accelerate

# ============================================================
# Import Libraries
# ============================================================

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ============================================================
# STEP 1 : CREATE KNOWLEDGE BASE
# ============================================================

documents = [

"""
Generative Artificial Intelligence is a branch of AI that creates
new content such as text, images, audio, video and computer programs.
""",

"""
Large Language Models are transformer-based models trained on massive
text datasets. They are used for text generation, summarization,
translation, question answering and conversational AI.
""",

"""
Retrieval-Augmented Generation (RAG) combines information retrieval
with text generation. It retrieves relevant documents from an external
knowledge base and provides them as context to a language model.
""",

"""
Vector databases store high-dimensional embeddings and perform
similarity search. Examples include FAISS, ChromaDB,
Pinecone, Weaviate and Milvus.
""",

"""
Prompt engineering is the process of designing clear instructions
that help language models generate accurate and useful responses.
Common techniques include zero-shot, few-shot and role-based prompting.
""",

"""
Fine-tuning adapts a pretrained language model to a specific task
or domain using a smaller domain-specific dataset.
"""

]

# ============================================================
# STEP 2 : LOAD EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ============================================================
# STEP 3 : CREATE DOCUMENT EMBEDDINGS
# ============================================================

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

document_embeddings = document_embeddings.astype("float32")

# Normalize vectors
faiss.normalize_L2(document_embeddings)

# ============================================================
# STEP 4 : CREATE FAISS VECTOR DATABASE
# ============================================================

embedding_dimension = document_embeddings.shape[1]

vector_database = faiss.IndexFlatIP(embedding_dimension)

vector_database.add(document_embeddings)

print("Knowledge Base Created Successfully!")

# ============================================================
# STEP 5 : LOAD TEXT GENERATION MODEL
# ============================================================

print("Loading FLAN-T5 Model...")

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("Model Loaded Successfully!")

# ============================================================
# STEP 6 : DOCUMENT RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    similarity_scores, indices = vector_database.search(
        query_embedding,
        top_k
    )

    retrieved_docs = []

    for index, score in zip(indices[0], similarity_scores[0]):

        retrieved_docs.append({

            "document": documents[index].strip(),

            "score": float(score)

        })

    return retrieved_docs

# ============================================================
# STEP 7 : ANSWER GENERATION FUNCTION
# ============================================================

def generate_answer(query, retrieved_documents):

    context = "\n\n".join(
        item["document"]
        for item in retrieved_documents
    )

    prompt = f"""
Answer the question using ONLY the information given below.

Context:
{context}

Question:
{query}

Instructions:
1. Give a clear answer.
2. Do not add extra information.
3. If the answer is unavailable, reply:
"The answer is not available in the knowledge base."

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )

    return result[0]["generated_text"]

# ============================================================
# STEP 8 : USER QUERY
# ============================================================

print("\n")
print("="*60)
print("RETRIEVAL AUGMENTED GENERATION (RAG)")
print("="*60)

user_query = input("\nEnter your question: ")

retrieved_results = retrieve_documents(
    user_query,
    top_k=2
)

answer = generate_answer(
    user_query,
    retrieved_results
)

# ============================================================
# STEP 9 : DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print("\n")
print("="*60)
print("RETRIEVED DOCUMENTS")
print("="*60)

for i, item in enumerate(retrieved_results, start=1):

    print(f"\nDocument {i}")

    print("-"*50)

    print(item["document"])

    print(f"\nSimilarity Score : {item['score']:.4f}")

# ============================================================
# STEP 10 : DISPLAY GENERATED ANSWER
# ============================================================

print("\n")
print("="*60)
print("GENERATED ANSWER")
print("="*60)

print(answer)

print("\n")
print("="*60)
print("PROGRAM EXECUTED SUCCESSFULLY")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 86.0 MB/s eta 0:00:00
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge Base Created Successfully!
Loading FLAN-T5 Model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"